# RSNA Knee: сборка «сырого» кэша на Kaggle

Ноутбук читает DICOM соревнования и сохраняет **по одному файлу на исследование**: `studies/<ID>.npz`.
Внутри для каждой серии лежат **все её срезы по порядку** (до 32), уже в одном масштабе и ориентации.
Всё остальное (какие срезы брать, нормализация, аугментации) делает класс `Dataset` в ноутбуках обучения.

## Что делается здесь и почему

Здесь только то, для чего нужны заголовки DICOM. Самих DICOM локально не будет, поэтому позже
этого уже не сделать:

| Шаг | Зачем |
|---|---|
| выбор серии для каждой пары «плоскость × подавление жира» | в исследовании 3-14 серий, берём по одной лучшей (больше срезов) на каждую из 6 комбинаций |
| порядок срезов по положению в теле | имена файлов случайные; порядок восстанавливается по `ImagePositionPatient` и `ImageOrientationPatient` |
| единая ориентация | разные аппараты кладут картинку по-разному (транспонировано, вверх ногами) |
| кроп 140 мм и ресайз до 224 | у всех исследований один физический масштаб; без `PixelSpacing` его не восстановить |
| яркость в 0-255 | у МРТ нет абсолютной шкалы; приводим серию к 0.5-99.5 перцентилям, чтобы хранить в uint8 |

Пустые комбинации **не** заполняются: подставлять пару (например, серию без fat-sat вместо отсутствующей
с fat-sat) будет `Dataset`.

## Две части

Kaggle сохраняет в Output не больше 20 ГБ за запуск, а все срезы весят больше. Поэтому исследования
делятся на `N_PARTS = 2` части (каждое второе по порядку ID), и ноутбук запускается дважды: с `PART = 0` и с `PART = 1`.
Каждый запуск становится отдельным датасетом.

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл. Input: соревнование `rsna-knee-abnormality-detection`.
2. Settings: Accelerator **None** (CPU), Internet можно выключить.
3. `PART = 0` -> **Save Version -> Save & Run All**. Когда досчитается: **Output -> New Dataset** `rsna-knee-raw-part0`.
4. Поменять на `PART = 1`, снова **Save & Run All**, потом **New Dataset** `rsna-knee-raw-part1`.
   Можно запустить вторую часть копией ноутбука параллельно с первой.

## Что получится и куда положить локально

```
data/raw/
├── studies/<StudyInstanceUID>.npz     4407 файлов из обеих частей
├── index_part0.csv, index_part1.csv   какая серия в каком слоте, сколько в ней срезов
├── rsna_raw_meta.json                 настройки сборки (IMG, CROP_MM, MAX_SLICES, слоты)
└── train.csv                          метки и тексты заключений
```

Содержимое обоих датасетов распаковывается в одну папку `data/raw/`: папки `studies/` сольются.

In [ ]:
import os 
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc 
import re
import math 
import time 
import glob 
import json
import hashlib 
import textwrap 
import warnings
from copy import deepcopy 
from pathlib import Path 
from concurrent.futures import ThreadPoolExecutor 

import cv2 
import numpy as np 
import pandas as pd 
import pydicom 
import matplotlib.pyplot as plt 
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import timm 
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
T0 = time.time()

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

try:
    display 
except NameError:
    display = print 

# Устройство: видеокарта NVIDIA (Kaggle) -> видеокарта Apple Silicon (MPS) -> процессор
DEV = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


def empty_device_cache():
    # освободить кэш памяти видеокарты; у CUDA и MPS это разные функции
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    elif DEV.type == "mps":
        torch.mps.empty_cache()
log(f"device: {DEV}, torch {torch.__version__}, timm {timm.__version__}, pydicom {pydicom.__version__}")

## Настройки

Меняются только `PART` (0 или 1) между двумя запусками. Остальное должно совпадать в обоих запусках.

In [ ]:
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/rsna-knee-abnormality-detection")
OUT_DIR = Path("/kaggle/working")       # всё, что здесь, станет содержимым датасета
STUDY_DIR = OUT_DIR / "studies"         # по файлу на исследование
STUDY_DIR.mkdir(parents=True, exist_ok=True)

N_PARTS = 2                # на сколько запусков делим исследования (лимит Output 20 ГБ)
PART = 0                   # какую часть собираем сейчас: 0 или 1

IMG = 224                  # сторона среза в пикселях
CROP_MM = 140.0            # квадрат 140x140 мм вокруг центра снимка -> один физический масштаб у всех
MAX_SLICES = 32            # если в серии больше срезов, берём 32 равномерно (у большинства серий 18-35)
SERIES_THREADS = 24        # сколько серий читаем параллельно (упор в диск Kaggle, а не в процессор)
CHUNK_STUDIES = 64         # сколько исследований обрабатываем за раз (ограничивает память)

log(f"DATA_DIR = {DATA_DIR} (существует: {DATA_DIR.exists()}), часть {PART} из {N_PARTS}")

## Какие серии берём

Шесть комбинаций «плоскость × подавление жира». Для каждой берём серию, где больше всего срезов
(минимум 3). Если комбинации в исследовании нет, её просто нет в файле.

In [ ]:
SLOTS = [  # (имя, плоскость, Fat_Suppression)
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]


def count_files(d):
    try:
        return sum(1 for e in os.scandir(d) if e.name.endswith(".dcm"))
    except OSError:
        return 0


def pick_series(series_df):
    # -> таблица «исследование, слот -> серия», одна строка на найденную комбинацию
    df = series_df.copy()
    df["dir"] = [str(DATA_DIR / "train_series" / str(a) / str(b))
                 for a, b in zip(df["StudyInstanceUID"], df["SeriesInstanceUID"])]
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        df["n_files"] = list(pool.map(count_files, df["dir"]))
    rows = []
    for study, g in df.groupby("StudyInstanceUID"):
        for k, (name, plane, fs) in enumerate(SLOTS):
            cand = g[(g["Anatomical_Plane"] == plane) & (g["Fat_Suppression"].astype(int) == fs)
                     & (g["n_files"] >= 3)]
            if len(cand):
                r = cand.sort_values("n_files", ascending=False).iloc[0]
                rows.append({"StudyInstanceUID": study, "slot": k, "slot_name": name,
                             "SeriesInstanceUID": r["SeriesInstanceUID"], "dir": r["dir"],
                             "plane": plane, "n_files": int(r["n_files"])})
    return pd.DataFrame(rows)

## Чтение одной серии

Функции заголовков, порядка срезов, ориентации и кропа те же, что в `02_report_labels_mil.ipynb`.
Новое только в `read_stack`: она читает **все** срезы серии (до `MAX_SLICES`), а не 12 выбранных.

1. Заголовки всех файлов -> сортировка по положению среза вдоль нормали к плоскости.
2. Если срезов больше `MAX_SLICES`, берём `MAX_SLICES` равномерно по стеку.
3. Декодирование; сломанный срез заменяется ближайшим целым.
4. Единая ориентация, перцентили 0.5-99.5 по всей серии -> [0, 1], кроп 140 мм, ресайз 224, uint8.

In [ ]:
HDR_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "InstanceNumber"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# Каноническая ориентация (из ноутбука 0.958). Для плоскости, у которой нормаль идёт вдоль оси
# 0 (x, лево-право) / 1 (y, перёд-зад) / 2 (z, верх-низ), задаём:
# (знак порядка срезов, (ось вдоль столбцов картинки, знак), (ось вдоль строк, знак))
# Система координат DICOM: x -> к ЛЕВОМУ боку пациента, y -> к спине, z -> к голове.
CANON = {
    0: (+1, (1, +1), (2, -1)),   # сагиттальная: слева направо по картинке = спереди назад, вниз = от головы
    1: (+1, (0, +1), (2, -1)),   # корональная: по картинке справа пациента налево, вниз = от головы
    2: (-1, (0, +1), (1, +1)),   # аксиальная: срезы сверху вниз, по картинке вниз = спереди назад
}


def _vec(v, n):
    try:
        a = np.asarray([float(x) for x in v], float)
        return a if len(a) >= n else None
    except Exception:
        return None


def read_headers(d):
    recs = []
    for f in sorted(os.listdir(d)):
        if not f.endswith(".dcm"):
            continue
        rec = {"file": f, "ipp": None, "iop": None, "ps": None, "inst": None}
        try:
            h = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True, force=True,
                                specific_tags=HDR_TAGS)
            rec["ipp"] = _vec(getattr(h, "ImagePositionPatient", []), 3)
            rec["iop"] = _vec(getattr(h, "ImageOrientationPatient", []), 6)
            rec["ps"] = _vec(getattr(h, "PixelSpacing", []), 2)
            rec["inst"] = float(getattr(h, "InstanceNumber", None) or 0)
        except Exception:
            pass
        recs.append(rec)
    return recs


def sort_slices(recs, plane_name):
    # -> (отсортированные записи, номер оси плоскости 0/1/2, IOP для ориентации)
    iop = next((r["iop"] for r in recs if r["iop"] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        ax = int(np.abs(normal).argmax())
        if all(r["ipp"] is not None for r in recs):
            sgn = CANON[ax][0] * np.sign(normal[ax])
            recs = sorted(recs, key=lambda r: float(np.dot(r["ipp"], normal)) * sgn)
            return recs, ax, iop
    else:
        ax = PLANES.index(plane_name) if plane_name in PLANES else 0
    return sorted(recs, key=lambda r: (r["inst"] or 0, r["file"])), ax, iop


def canon_orient(img, iop, ps, ax):
    # поворачиваем/отражаем срез к канонической ориентации; ps (размер пикселя) меняем вместе с ним
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[ax]
    if abs(r[cax]) < abs(r[rax]):           # строки и столбцы перепутаны -> транспонируем
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]                  # отражаем по горизонтали
    if c[rax] * rsgn < 0:
        img = img[::-1]                     # отражаем по вертикали
    return np.ascontiguousarray(img), ps


def crop_resize(img, ps):
    # квадрат CROP_MM мм вокруг центра -> IMG×IMG; если снимок меньше, дополняем нулями
    out = np.zeros((IMG, IMG), np.float32)
    mm_per_px = CROP_MM / IMG
    h, w = img.shape
    ch = min(h, int(round(CROP_MM / ps[0])))
    cw = min(w, int(round(CROP_MM / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(IMG, int(round(ch * ps[0] / mm_per_px))))
    nw = max(1, min(IMG, int(round(cw * ps[1] / mm_per_px))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR   # AREA - без «лесенки» при уменьшении
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (IMG - nh) // 2, (IMG - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def decode(path):
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array.astype(np.float32)
    if a.ndim == 3:                         # редкий «мультикадровый» файл - берём средний кадр
        a = a[len(a) // 2]
    a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a                     # в MONOCHROME1 «белое» и «чёрное» перевёрнуты
    return a


def read_stack(d, plane_name):
    # серия -> uint8 [n_срезов, IMG, IMG] по порядку в теле, или None, если ничего не прочиталось
    recs = read_headers(d)
    if not recs:
        return None
    recs, ax, iop = sort_slices(recs, plane_name)
    n = len(recs)
    idx = np.unique(np.linspace(0, n - 1, min(n, MAX_SLICES)).round().astype(int))

    imgs = {}
    for i in idx:
        try:
            imgs[int(i)] = decode(os.path.join(d, recs[int(i)]["file"]))
        except Exception:
            imgs[int(i)] = None
    ok = [i for i, a in imgs.items() if a is not None]
    if not ok:
        return None
    for i in imgs:                          # сломанный срез -> ближайший целый
        if imgs[i] is None:
            imgs[i] = imgs[min(ok, key=lambda j: abs(j - i))]

    ps_default = next((r["ps"] for r in recs if r["ps"] is not None), None)
    planes = []
    for i in idx:
        a = imgs[int(i)]
        ps = recs[int(i)]["ps"] if recs[int(i)]["ps"] is not None else ps_default
        if ps is None:                      # нет размера пикселя -> считаем, что снимок = кропу
            ps = np.array([CROP_MM / max(a.shape)] * 2)
        planes.append(canon_orient(a, iop, list(ps), ax))

    allpx = np.concatenate([a[::4, ::4].ravel() for a, _ in planes])
    lo, hi = np.percentile(allpx, [0.5, 99.5])
    stack = np.stack([crop_resize(np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1), ps) for a, ps in planes])
    return (stack * 255).round().astype(np.uint8)

## Сборка

Исследования этой части обрабатываются кусками по `CHUNK_STUDIES`: серии куска читаются параллельно, затем
каждое исследование сохраняется в свой сжатый файл. В логе - скорость, размер и прогноз итогового размера.

In [ ]:
import shutil

train_df = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
studies = sorted(train_df["StudyInstanceUID"].astype(str))
part_studies = studies[PART::N_PARTS]                     # каждое N_PARTS-е исследование
series_part = train_series[train_series["StudyInstanceUID"].astype(str).isin(set(part_studies))]

INDEX = pick_series(series_part)
INDEX["StudyInstanceUID"] = INDEX["StudyInstanceUID"].astype(str)
log(f"часть {PART}: {len(part_studies)} исследований, {len(INDEX)} серий в слотах")

t0, done, n_slices_saved, failed = time.time(), 0, {}, []
for c0 in range(0, len(part_studies), CHUNK_STUDIES):
    chunk = part_studies[c0:c0 + CHUNK_STUDIES]
    jobs = INDEX[INDEX["StudyInstanceUID"].isin(set(chunk))]
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        stacks = list(pool.map(read_stack, jobs["dir"], jobs["plane"]))
    by_study = {}
    for r, st in zip(jobs.itertuples(index=False), stacks):
        if st is None:
            failed.append(r.SeriesInstanceUID)
            continue
        by_study.setdefault(r.StudyInstanceUID, {})[f"slot{r.slot}"] = st
        n_slices_saved[(r.StudyInstanceUID, r.slot)] = len(st)
    for study in chunk:
        np.savez_compressed(STUDY_DIR / f"{study}.npz", **by_study.get(study, {}))
    done += len(chunk)
    size_gb = sum(f.stat().st_size for f in STUDY_DIR.iterdir()) / 1024 ** 3
    rate = done / max(time.time() - t0, 1e-6)
    log(f"  {done}/{len(part_studies)} исследований, {rate:.2f} иссл/с, осталось ~{(len(part_studies) - done) / rate / 60:.0f} мин, "
        f"{size_gb:.2f} ГБ (прогноз на часть ~{size_gb / done * len(part_studies):.1f} ГБ)")

INDEX["n_slices_saved"] = [n_slices_saved.get((s, k), 0) for s, k in zip(INDEX["StudyInstanceUID"], INDEX["slot"])]
INDEX.drop(columns=["dir"]).to_csv(OUT_DIR / f"index_part{PART}.csv", index=False)
with open(OUT_DIR / "rsna_raw_meta.json", "w") as fh:
    json.dump({"IMG": IMG, "CROP_MM": CROP_MM, "MAX_SLICES": MAX_SLICES, "N_PARTS": N_PARTS,
               "SLOTS": [list(s) for s in SLOTS], "intensity": "percentile 0.5-99.5 per series -> uint8"}, fh, indent=1)
shutil.copy(DATA_DIR / "train.csv", OUT_DIR / "train.csv")
log(f"готово: {done} файлов, не прочитались {len(failed)} серий, "
    f"всего {sum(f.stat().st_size for f in STUDY_DIR.iterdir()) / 1024 ** 3:.2f} ГБ")

## Проверка глазами

Одно исследование: центральный срез каждого найденного слота и число срезов в нём.

In [ ]:
example = np.load(STUDY_DIR / f"{part_studies[len(part_studies) // 2]}.npz")
keys = sorted(example.files)
fig, axes = plt.subplots(1, len(keys), figsize=(3 * len(keys), 3.2))
for ax, key in zip(np.atleast_1d(axes), keys):
    st = example[key]
    ax.imshow(st[len(st) // 2], cmap="gray")
    ax.set_title(f"{SLOTS[int(key[4:])][0]}: {len(st)} срезов", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()